# E2C — precision audit of the cospectral separations

The cospectral-mate separations (L1 1e-8 to 1e-5 at n=14) are the paper's
evidence that QuIC carries non-spectral content. Those numbers came from one
float64 pipeline. E2C certifies them with independent machinery and extended
precision, closing the four remaining E2C bullets (the tolerance bullet was
closed by E8A's exact integer census):

1. **Independent statevector recompute** — a pure-numpy circuit, no qiskit,
   ported from the producer's `QuIK_circuit` source (RX degree-encoder →
   RZZ entangler → RX mixer, r=1; NOT the earlier "ZZ → Z → RX"
   reconstruction, which was wrong). **Load-bearing validation gate**: hard
   assert max|independent − stored| < 1e-12 over all 89 cospectral group
   members before anything downstream runs. Rehearsed offline against the
   producer's qiskit pipeline at ~1e-15 on random cubics.
2. **Extended precision** — longdouble (80-bit) recompute of all 89 members,
   with the float64 reconstruction floor *measured* per graph (not assumed);
   mpmath dps=40 recompute for every pair separated by < 1e-6, certifying
   longdouble in turn.
3. **Absolute/relative separation report** — per-pair L1 in three
   arithmetics, a ×floor column (pair separation over the summed member
   floors), and standing against the census-wide pairwise-L1 scale.
4. **Head/tail decomposition** — fraction of each pair's L1 carried by the
   top-50/top-90 sorted coordinates, and the coordinate depth at which the
   cumulative separation reaches 50% and 90% (the head-weighted metric
   doctrine, per pair).

**Pre-registered expectations.** Gate passes (a failure is a port bug — diff
`quic_probs` against the producer's `QuIK_circuit`, fix here, never touch the
producer). Float64 floors land ~1e-16–1e-14 per graph; every known pair
separation sits orders of magnitude above its floor, so the separations are
physics, not float artifacts — the ×floor column is the paper sentence.
mpmath confirms longdouble to ≥ 15 relative digits (offline: ld floor
~9e-19). Head fractions high / k50 small, per the truncation-tracer
mechanism. If any pair's ×floor < 10, that pair is unresolved at float64 and
its mpmath value becomes the number of record — flagged, not hidden.

**Runtime.** Gate + longdouble pass: minutes. mpmath is the long pole:
~15 s/graph at n=16, worst case all 83 members ≈ 20 min. n=16 census
distance scale (sampled pairs): minutes. Config: canonical angles
(2.875, 2.0, 0.1), r=1, SEED=0 — locked, as everywhere.

## Environment

In [1]:
# qiskit is needed ONLY to unpickle the producer data_dicts (they embed
# QuantumCircuit objects). The independent recompute deliberately imports
# nothing from it.
!pip install qiskit --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 63.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 47.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 2.6 MB/s eta 0:00:00


In [2]:
import pickle
import time

import numpy as np
import networkx as nx

from collections import defaultdict
from itertools import combinations

from mpmath import mp, mpf, mpc, fsum

from tqdm.notebook import tqdm

In [3]:
SEED = 0
NS = (14, 16)
BASES = {
    14: "/kaggle/input/notebooks/lukemiller1987/n14-aaai-dataset/",
    16: "/kaggle/input/notebooks/lukemiller1987/n16-aaai-dataset/",
}
EXPECTED_CUBIC_COUNTS = {14: 509, 16: 4060}

# Circuit parameters — canonical, locked (producer values, verbatim).
MAX_ENCODING_ROTATION = 2.875
ENTANGLER_SCALAR = 2.0
MIXER_SCALAR = 0.1
REPS = 1

GATE_TOL = 1e-12        # hard assert: independent f64 vs stored, per coordinate
MP_THRESHOLD = 1e-6     # pairs below this longdouble L1 get the mpmath pass
MP_DPS = 40
N16_SAMPLE_PAIRS = 300_000   # census L1 scale at n=16 (full pdist is infeasible)

## Load both datasets

Only `adj_mat`, `graph6`, and the stored `exact_vector` are consumed.
Integrity asserts: enumeration counts, sortedness (descending) and
normalization of the stored vectors.

In [4]:
def load_dataset(n):
    with open(BASES[n] + f"n{n}_data_dict.pkl", 'rb') as f:
        data_dict = pickle.load(f)
    keys = sorted(data_dict)
    assert len(keys) == EXPECTED_CUBIC_COUNTS[n], (
        f'expected {EXPECTED_CUBIC_COUNTS[n]} graphs at n={n}, got {len(keys)}')
    A = np.stack([data_dict[k]['adj_mat'] for k in keys]).astype(np.int64)
    V = np.vstack([data_dict[k]['exact_vector'] for k in keys])
    assert V.shape == (len(keys), 1 << n)
    assert np.all(np.diff(V, axis=1) <= 0), 'stored vectors not sorted descending'
    norm_err = np.abs(V.sum(axis=1) - 1.0).max()
    assert norm_err < 1e-12, f'stored vectors not normalized: {norm_err:.2e}'
    print(f'n={n}: {len(keys)} graphs; stored-vector normalization max err {norm_err:.2e}')
    return {'keys': keys, 'A': A, 'V': V}

DATA = {n: load_dataset(n) for n in NS}

n=14: 509 graphs; stored-vector normalization max err 2.00e-15
n=16: 4060 graphs; stored-vector normalization max err 2.66e-15


## Cospectral census — recomputed, asserted against E8A

Exact int64 trace tuples (E8A's method, seconds — recompute rather than
cross-load). Hard asserts pin the census to E8A's certified numbers:
3 groups / 6 graphs at n=14; 41 groups / 83 graphs at n=16 with group
sizes {2: 40, 3: 1}; 89 members and 46 within-group pairs total.

In [5]:
GROUPS, MEMBERS, PAIRS = {}, {}, {}
for n in NS:
    A = DATA[n]['A']
    B = len(A)
    traces = np.zeros((B, n), dtype=np.int64)
    P = A.copy()
    traces[:, 0] = np.trace(P, axis1=1, axis2=2)
    for k in range(1, n):
        P = P @ A                       # exact int64; walk counts < n * 3^n
        traces[:, k] = np.trace(P, axis1=1, axis2=2)
    exact = defaultdict(list)
    for i in range(B):
        exact[tuple(traces[i])].append(i)
    GROUPS[n] = sorted([sorted(v) for v in exact.values() if len(v) > 1])
    MEMBERS[n] = sorted({i for g in GROUPS[n] for i in g})
    PAIRS[n] = [(i, j) for g in GROUPS[n] for i, j in combinations(g, 2)]
    print(f'n={n}: {len(GROUPS[n])} groups, {len(MEMBERS[n])} members, '
          f'{len(PAIRS[n])} within-group pairs')

# pin to the E8A-certified census
assert len(GROUPS[14]) == 3 and len(MEMBERS[14]) == 6
assert len(GROUPS[16]) == 41 and len(MEMBERS[16]) == 83
from collections import Counter
assert Counter(len(g) for g in GROUPS[16]) == Counter({2: 40, 3: 1})
assert [582, 3061, 3800] in GROUPS[16], 'the mutually cospectral triple'
assert len(MEMBERS[14]) + len(MEMBERS[16]) == 89
assert len(PAIRS[14]) + len(PAIRS[16]) == 46
print('census matches E8A exactly')

n=14: 3 groups, 6 members, 3 within-group pairs
n=16: 41 groups, 83 members, 43 within-group pairs
census matches E8A exactly


## Independent circuit — pure numpy, dtype-parameterized

Ported from the producer's `QuIK_circuit` (both dataset notebooks,
identical): RX(θ_enc·deg_q) per qubit → RZZ(2.0) per edge → RX(0.1) per
qubit, r=1, measure, probabilities, sort descending. Qiskit conventions
reproduced: RX(θ) = exp(−iθX/2) = [[cos θ/2, −i sin θ/2], [−i sin θ/2,
cos θ/2]]; RZZ(θ) = exp(−iθ/2·Z⊗Z), diagonal phase e^{∓iθ/2} for
equal/unequal bits. Qubit order is irrelevant after sorting (permutation
invariance is a smoke test below).

Two deliberate choices:

- **Angles are the float64 numbers the producer used** (encoding_scalar =
  2.875/max_degree computed in f64), then lifted exactly into the working
  precision. Extended precision applies to the arithmetic, not to
  reinterpreting the circuit — otherwise the "floor" would conflate
  rounding with a genuinely different circuit.
- **The entangler is applied as a phase LUT** over m = number of
  disagreeing edges per basis state (m is exact integer arithmetic;
  Σ_edges ±1 = E − 2m), so the diagonal costs E+1 trig evaluations instead
  of 2^n·E.

`quic_probs(A, dtype)` serves float64 (complex128) and longdouble
(clongdouble); `quic_probs_mp(A)` is the same circuit in mpmath at
dps=40.

In [6]:
def circuit_angles_f64(adj):
    """The angles exactly as the producer computed them, in float64."""
    deg = np.asarray(adj, dtype=np.float64).sum(axis=1)
    enc_scalar = np.float64(MAX_ENCODING_ROTATION) / deg.max()
    return enc_scalar * deg, np.float64(ENTANGLER_SCALAR), np.float64(MIXER_SCALAR)


def disagree_counts(adj):
    """m[s] = number of edges whose endpoint bits differ in basis state s. Exact."""
    n = adj.shape[0]
    eu, ev = np.nonzero(np.triu(adj, k=1))
    idx = np.arange(1 << n, dtype=np.int64)
    m = np.zeros(1 << n, dtype=np.int64)
    for u, v in zip(eu, ev):
        m += ((idx >> u) & 1) ^ ((idx >> v) & 1)
    return m, len(eu)


def quic_probs(adj, dtype=np.complex128):
    """Sorted (descending) exact output distribution of the QuIC circuit."""
    real = np.float64 if dtype == np.complex128 else np.longdouble
    mi = dtype(-1j)
    adj = np.asarray(adj)
    n = adj.shape[0]
    enc, th_ent, th_mix = circuit_angles_f64(adj)

    # encoder: product state; qubit q lives at bit position q of the index
    psi = np.ones(1, dtype=dtype)
    for q in range(n):
        half = real(enc[q]) / 2
        psi = np.concatenate([psi * np.cos(half), psi * (mi * np.sin(half))])

    # entangler: diagonal phase, LUT over disagreeing-edge count
    m, E = disagree_counts(adj)
    ang = -(real(th_ent) / 2) * (E - 2 * np.arange(E + 1)).astype(real)
    psi = psi * (np.cos(ang) + dtype(1j) * np.sin(ang))[m]

    # mixer: RX(th_mix) on every qubit
    half = real(th_mix) / 2
    c, s = np.cos(half), np.sin(half)
    mis = mi * s
    for q in range(n):
        psi = psi.reshape(-1, 2, 1 << q)
        a0 = psi[:, 0, :].copy()
        a1 = psi[:, 1, :].copy()
        psi[:, 0, :] = c * a0 + mis * a1
        psi[:, 1, :] = mis * a0 + c * a1
        psi = psi.reshape(-1)

    probs = psi.real ** 2 + psi.imag ** 2
    return np.sort(probs)[::-1]


def quic_probs_mp(adj, dps=MP_DPS):
    """Same circuit, same f64 angles, mpmath arithmetic at `dps` digits."""
    mp.dps = dps
    adj = np.asarray(adj)
    n = adj.shape[0]
    N = 1 << n
    enc, th_ent, th_mix = circuit_angles_f64(adj)

    psi = [mpc(1)]
    for q in range(n):
        half = mpf(float(enc[q])) / 2
        c, mis = mp.cos(half), mpc(0, -1) * mp.sin(half)
        psi = [p * c for p in psi] + [p * mis for p in psi]

    m, E = disagree_counts(adj)
    lut = []
    for k in range(E + 1):
        ang = -(mpf(float(th_ent)) / 2) * (E - 2 * k)
        lut.append(mpc(mp.cos(ang), mp.sin(ang)))
    psi = [psi[i] * lut[m[i]] for i in range(N)]

    half = mpf(float(th_mix)) / 2
    c, mis = mp.cos(half), mpc(0, -1) * mp.sin(half)
    for q in range(n):
        step = 1 << q
        block = step << 1
        for base in range(0, N, block):
            for i0 in range(base, base + step):
                i1 = i0 + step
                a0, a1 = psi[i0], psi[i1]
                psi[i0] = c * a0 + mis * a1
                psi[i1] = mis * a0 + c * a1

    return sorted((p.real ** 2 + p.imag ** 2 for p in psi), reverse=True)


def ld_to_mpf(x):
    """Exact lift of an 80-bit longdouble into mpmath via a hi+lo f64 split."""
    hi = np.float64(x)
    lo = np.float64(np.longdouble(x) - np.longdouble(hi))
    return mpf(float(hi)) + mpf(float(lo))

## Smoke tests

Normalization and permutation invariance in longdouble on one census
member per n. Offline values on random cubics: normalization ~9e-19,
permutation invariance ~3e-20.

In [7]:
rng = np.random.default_rng(SEED)
for n in NS:
    k = MEMBERS[n][0]
    A = DATA[n]['A'][k].astype(np.float64)
    vld = quic_probs(A, np.clongdouble)
    perm = rng.permutation(n)
    vperm = quic_probs(A[np.ix_(perm, perm)], np.clongdouble)
    norm_err = abs(vld.sum() - np.longdouble(1))
    perm_err = np.abs(vld - vperm).sum()
    print(f'n={n} graph {k}:  |Σp − 1| = {float(norm_err):.2e}   '
          f'perm-invariance L1 = {float(perm_err):.2e}')
    assert norm_err < 1e-15 and perm_err < 1e-15

n=14 graph 79:  |Σp − 1| = 9.76e-19   perm-invariance L1 = 7.90e-20
n=16 graph 1:  |Σp − 1| = 4.34e-19   perm-invariance L1 = 1.31e-19


## Validation gate — load-bearing

Independent float64 recompute of all 89 members, hard-asserted against the
stored vectors coordinate-wise at 1e-12. Nothing downstream is meaningful
if this fails. On failure: the bug is in `quic_probs` — diff it against the
producer's `QuIK_circuit` (RX encoder → RZZ → RX mixer, qiskit half-angle
conventions); never touch the producer.

In [8]:
V64 = {n: {} for n in NS}
worst_overall = 0.0
for n in NS:
    worst_n = 0.0
    for k in tqdm(MEMBERS[n], desc=f'n={n} gate'):
        v = quic_probs(DATA[n]['A'][k].astype(np.float64), np.complex128)
        V64[n][k] = v
        d = np.abs(v - DATA[n]['V'][k]).max()
        assert d < GATE_TOL, f'GATE FAILED n={n} graph {k}: max coord diff {d:.2e}'
        worst_n = max(worst_n, d)
    worst_overall = max(worst_overall, worst_n)
    print(f'n={n}: worst |independent − stored| = {worst_n:.2e}')
print(f'\nGATE PASSED — worst coordinate deviation {worst_overall:.2e} < {GATE_TOL:.0e}')

n=14 gate:   0%|          | 0/6 [00:00<?, ?it/s]

n=14: worst |independent − stored| = 1.89e-15


n=16 gate:   0%|          | 0/83 [00:00<?, ?it/s]

n=16: worst |independent − stored| = 2.22e-15

GATE PASSED — worst coordinate deviation 2.22e-15 < 1e-12


## Longdouble recompute + measured float64 floors

All 89 members in 80-bit longdouble. The float64 reconstruction floor is
*measured* per graph as L1(sorted f64, sorted longdouble) — the actual
accumulated rounding of the f64 pipeline on that graph, not an eps-count
estimate.

In [9]:
VLD = {n: {} for n in NS}
FLOOR64 = {n: {} for n in NS}
for n in NS:
    for k in tqdm(MEMBERS[n], desc=f'n={n} longdouble'):
        vld = quic_probs(DATA[n]['A'][k].astype(np.float64), np.clongdouble)
        VLD[n][k] = vld
        FLOOR64[n][k] = float(np.abs(vld - V64[n][k]).sum())
    fl = np.array([FLOOR64[n][k] for k in MEMBERS[n]])
    print(f'n={n}: f64 floor per graph — min {fl.min():.2e}  '
          f'median {np.median(fl):.2e}  max {fl.max():.2e}')

n=14 longdouble:   0%|          | 0/6 [00:00<?, ?it/s]

n=14: f64 floor per graph — min 7.44e-17  median 3.49e-16  max 5.82e-16


n=16 longdouble:   0%|          | 0/83 [00:00<?, ?it/s]

n=16: f64 floor per graph — min 9.05e-17  median 5.72e-16  max 1.11e-15


## Pair separations + mpmath pass

Pair L1 in float64 (stored vectors — the numbers the rest of the paper
quotes) and longdouble (independent). Every pair with longdouble L1 below
1e-6 gets the mpmath dps=40 recompute; member vectors are cached so shared
members (the triple) compute once. The longdouble floor is measured the
same way the f64 floor was: L1(longdouble, mpmath) per graph.

In [10]:
PAIR_ROWS = []
for n in NS:
    for (i, j) in PAIRS[n]:
        l1_f64 = float(np.abs(DATA[n]['V'][i] - DATA[n]['V'][j]).sum())
        l1_ld = float(np.abs(VLD[n][i] - VLD[n][j]).sum())
        PAIR_ROWS.append({'n': n, 'i': i, 'j': j,
                          'l1_f64': l1_f64, 'l1_ld': l1_ld,
                          'floor64': FLOOR64[n][i] + FLOOR64[n][j]})

mp_pairs = [r for r in PAIR_ROWS if r['l1_ld'] < MP_THRESHOLD]
mp_graphs = sorted({(r['n'], r[k]) for r in mp_pairs for k in ('i', 'j')})
print(f'{len(mp_pairs)} of {len(PAIR_ROWS)} pairs below {MP_THRESHOLD:.0e} '
      f'-> mpmath pass over {len(mp_graphs)} graphs (~15 s each at n=16)')

VMP, FLOOR_LD = {}, {}
for (n, k) in tqdm(mp_graphs, desc='mpmath dps=40'):
    t0 = time.time()
    vmp = quic_probs_mp(DATA[n]['A'][k].astype(np.float64))
    VMP[(n, k)] = vmp
    lifted = [ld_to_mpf(x) for x in VLD[n][k]]
    FLOOR_LD[(n, k)] = float(fsum(abs(a - b) for a, b in zip(vmp, lifted)))
    print(f'  n={n} graph {k}: ld floor {FLOOR_LD[(n, k)]:.2e}  ({time.time()-t0:.0f}s)')

for r in PAIR_ROWS:
    key_i, key_j = (r['n'], r['i']), (r['n'], r['j'])
    if key_i in VMP and key_j in VMP:
        r['l1_mp'] = float(fsum(abs(a - b)
                                for a, b in zip(VMP[key_i], VMP[key_j])))
        r['floor_ld'] = FLOOR_LD[key_i] + FLOOR_LD[key_j]
    else:
        r['l1_mp'] = None
        r['floor_ld'] = None

30 of 46 pairs below 1e-06 -> mpmath pass over 60 graphs (~15 s each at n=16)


mpmath dps=40:   0%|          | 0/60 [00:00<?, ?it/s]

  n=14 graph 79: ld floor 1.03e-18  (3s)
  n=14 graph 234: ld floor 8.51e-19  (3s)
  n=14 graph 239: ld floor 8.50e-19  (3s)
  n=14 graph 458: ld floor 7.19e-19  (3s)
  n=16 graph 24: ld floor 6.78e-19  (15s)
  n=16 graph 35: ld floor 4.63e-19  (15s)
  n=16 graph 82: ld floor 6.67e-19  (15s)
  n=16 graph 83: ld floor 7.74e-19  (15s)
  n=16 graph 259: ld floor 5.72e-19  (15s)
  n=16 graph 268: ld floor 6.18e-19  (15s)
  n=16 graph 282: ld floor 7.83e-19  (15s)
  n=16 graph 320: ld floor 7.10e-19  (15s)
  n=16 graph 476: ld floor 6.73e-19  (15s)
  n=16 graph 479: ld floor 6.37e-19  (15s)
  n=16 graph 582: ld floor 6.56e-19  (15s)
  n=16 graph 583: ld floor 6.35e-19  (15s)
  n=16 graph 594: ld floor 6.35e-19  (15s)
  n=16 graph 601: ld floor 6.44e-19  (15s)
  n=16 graph 817: ld floor 7.82e-19  (15s)
  n=16 graph 884: ld floor 6.70e-19  (15s)
  n=16 graph 885: ld floor 7.20e-19  (15s)
  n=16 graph 967: ld floor 6.85e-19  (15s)
  n=16 graph 1034: ld floor 6.60e-19  (15s)
  n=16 graph 1035: 

## Census distance scale

Standing of each pair against the census-wide pairwise-L1 distribution:
full pdist at n=14 (129,286 pairs); a seeded 300k random-pair sample at
n=16 (full pdist over 4060 × 65536-dim vectors is infeasible). Stored f64
vectors — the same arithmetic the standing is quoted against elsewhere.

In [11]:
from scipy.spatial.distance import pdist

CENSUS_SCALE = {}
d14 = pdist(DATA[14]['V'], metric='cityblock')
CENSUS_SCALE[14] = {'dists': np.sort(d14), 'kind': 'exhaustive',
                    'median': float(np.median(d14))}
print(f'n=14: {len(d14)} pairs (exhaustive), median L1 {CENSUS_SCALE[14]["median"]:.4f}')

rng = np.random.default_rng(SEED)
B = len(DATA[16]['V'])
ii = rng.integers(0, B, N16_SAMPLE_PAIRS)
jj = rng.integers(0, B, N16_SAMPLE_PAIRS)
keep = ii != jj
ii, jj = ii[keep], jj[keep]
d16 = np.empty(len(ii))
CH = 500
for a in tqdm(range(0, len(ii), CH), desc='n=16 sampled pairs'):
    b = min(a + CH, len(ii))
    d16[a:b] = np.abs(DATA[16]['V'][ii[a:b]] - DATA[16]['V'][jj[a:b]]).sum(axis=1)
CENSUS_SCALE[16] = {'dists': np.sort(d16), 'kind': f'sampled ({len(ii)} pairs, seed {SEED})',
                    'median': float(np.median(d16))}
print(f'n=16: {len(d16)} sampled pairs, median L1 {CENSUS_SCALE[16]["median"]:.4f}')

for r in PAIR_ROWS:
    ds = CENSUS_SCALE[r['n']]['dists']
    r['pctile'] = float(np.searchsorted(ds, r['l1_f64']) / len(ds) * 100)
    r['rel_median'] = r['l1_f64'] / CENSUS_SCALE[r['n']]['median']

n=14: 129286 pairs (exhaustive), median L1 0.0001


n=16 sampled pairs:   0%|          | 0/600 [00:00<?, ?it/s]

n=16: 299918 sampled pairs, median L1 0.0001


## Head/tail decomposition + per-pair report

Coordinate-wise |Δ| between the two sorted longdouble vectors, cumulated
head-first (coordinate order = sorted-probability rank — the doctrine's
head). Columns: L1 in three arithmetics; ×floor = longdouble L1 over the
summed measured f64 floors of the pair (the certification number — how far
above float64 rounding the separation sits); census percentile and ratio
to census median; hf50/hf90 = fraction of the pair's L1 in the top-50 /
top-90 coordinates; k50/k90 = coordinate depth where the cumulative
separation reaches 50% / 90%.

In [12]:
for r in PAIR_ROWS:
    n, i, j = r['n'], r['i'], r['j']
    d = np.abs(VLD[n][i] - VLD[n][j])
    tot = d.sum()
    cum = np.cumsum(d) / tot
    r['hf50'] = float(cum[49])
    r['hf90'] = float(cum[89])
    r['k50'] = int(np.argmax(cum >= 0.5) + 1)
    r['k90'] = int(np.argmax(cum >= 0.9) + 1)
    r['xfloor'] = r['l1_ld'] / r['floor64']

hdr = (f'{"n":>3} {"i":>5} {"j":>5} | {"L1 f64":>10} {"L1 ld":>10} {"L1 mp40":>10} '
       f'{"floor64":>9} {"xfloor":>9} | {"pctile":>7} {"x median":>9} | '
       f'{"hf50":>6} {"hf90":>6} {"k50":>5} {"k90":>5}')
print(hdr)
print('-' * len(hdr))
for r in sorted(PAIR_ROWS, key=lambda r: (r['n'], r['l1_ld'])):
    mp_s = f"{r['l1_mp']:.3e}" if r['l1_mp'] is not None else '      --  '
    flag = '  <-- UNRESOLVED at f64' if r['xfloor'] < 10 else ''
    print(f"{r['n']:>3} {r['i']:>5} {r['j']:>5} | {r['l1_f64']:.3e} {r['l1_ld']:.3e} "
          f"{mp_s:>10} {r['floor64']:.2e} {r['xfloor']:9.2e} | "
          f"{r['pctile']:7.2f} {r['rel_median']:.2e} | "
          f"{r['hf50']:6.3f} {r['hf90']:6.3f} {r['k50']:>5} {r['k90']:>5}{flag}")

n_unres = sum(1 for r in PAIR_ROWS if r['xfloor'] < 10)
print(f'\npairs within 10x of the measured f64 floor: {n_unres} of {len(PAIR_ROWS)}')

  n     i     j |     L1 f64      L1 ld    L1 mp40   floor64    xfloor |  pctile  x median |   hf50   hf90   k50   k90
----------------------------------------------------------------------------------------------------------------------
 14   234   239 | 5.868e-08 5.868e-08  5.868e-08 4.32e-16  1.36e+08 |    0.00 6.91e-04 |  0.335  0.460   115  1133
 14    79   458 | 3.908e-07 3.908e-07  3.908e-07 9.48e-16  4.12e+08 |    0.06 4.60e-03 |  0.295  0.320   338   560
 14   201   203 | 1.692e-05 1.692e-05       --   5.82e-16  2.91e+10 |    5.82 1.99e-01 |  0.668  0.856    42   167
 16   594   601 | 1.917e-08 1.917e-08  1.917e-08 7.74e-16  2.48e+07 |    0.00 2.49e-04 |  0.053  0.224   295  1432
 16  2428  2431 | 2.719e-08 2.719e-08  2.719e-08 8.73e-16  3.12e+07 |    0.00 3.54e-04 |  0.092  0.134   875  1995
 16   282  1494 | 3.403e-08 3.403e-08  3.403e-08 1.29e-15  2.65e+07 |    0.00 4.43e-04 |  0.091  0.168   497  2804
 16   479   967 | 4.104e-08 4.104e-08  4.104e-08 5.78e-16  7.10e+07 |   

In [13]:
out = {
    'config': {'max_encoding_rotation': MAX_ENCODING_ROTATION,
               'entangler_scalar': ENTANGLER_SCALAR,
               'mixer_scalar': MIXER_SCALAR, 'reps': REPS,
               'gate_tol': GATE_TOL, 'mp_threshold': MP_THRESHOLD,
               'mp_dps': MP_DPS, 'seed': SEED},
    'groups': GROUPS, 'members': MEMBERS,
    'gate_worst': worst_overall,
    'floor64': FLOOR64,
    'floor_ld': FLOOR_LD,
    'pair_rows': PAIR_ROWS,
    'census_scale': {n: {'kind': CENSUS_SCALE[n]['kind'],
                         'median': CENSUS_SCALE[n]['median'],
                         'quantiles': {q: float(np.quantile(CENSUS_SCALE[n]['dists'], q))
                                       for q in (0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99)}}
                     for n in NS},
}
with open('/kaggle/working/e2c_precision_audit.pkl', 'wb') as f:
    pickle.dump(out, f)
print('saved e2c_precision_audit.pkl')

saved e2c_precision_audit.pkl


## Results

(Written after the run. The reading order: (1) the gate line — if it
printed, the independent recompute certifies the stored vectors and every
downstream number in the paper that touches them; (2) the ×floor column —
separations orders of magnitude above the measured f64 floor are the
one-sentence answer to "are the 1e-8 separations just float noise"; any
UNRESOLVED flags promote that pair's mpmath value to the number of record;
(3) the ld-floor prints from the mpmath pass — they certify longdouble the
same way longdouble certified f64; (4) hf/k columns against the
truncation-tracer mechanism: the head should carry the separation.
Nothing here feeds a claim beyond precision certification — the standing
columns restate E2's cospectral analysis on certified arithmetic, they do
not replace it.)

## E2C results — the certification, and the one sentence it buys

**The gate passed with three orders of headroom.** The independent
pure-numpy pipeline reproduced every stored vector across all 89
cospectral group members to a worst coordinate deviation of 1.89e-15 at
n=14 and 2.22e-15 at n=16, against a gate of 1e-12. Two independent
implementations of the circuit now agree at the float64 noise scale, so
every number in the paper that touches the stored vectors rests on a
verified reconstruction rather than a single pipeline. The smoke tests
matched their offline rehearsal values (normalization 4.3e-19 to 9.8e-19,
permutation invariance 7.9e-20 to 1.3e-19 in longdouble).

**The floors landed where pre-registered, and they were measured, not
assumed.** The per-graph float64 reconstruction floor spans 7.4e-17 to
1.1e-15 (medians 3.5e-16 at n=14 and 5.7e-16 at n=16), inside the
pre-registered 1e-16 to 1e-14 band. The mpmath pass then certified
longdouble the same way longdouble certified float64: all 60 recomputed
members returned longdouble floors between 4.6e-19 and 1.0e-18, matching
the ~9e-19 offline value.

**Zero pairs are unresolved — the pre-registered escape hatch was never
needed.** The certification number is the minimum of the ×floor column:
the smallest separation in either census, pair (594, 601) at n=16 with
L1 = 1.917e-08, exceeds its own pair's measured float64 floor by a factor
of 2.48e+07. The column maximum is 2.91e+10 for (201, 203). The paper
sentence follows directly: every cospectral separation exceeds the
measured float64 rounding floor of its own reconstruction by at least
seven orders of magnitude, so the separations are properties of the
circuit, not artifacts of the arithmetic. On the 30 pairs below 1e-6, the
mpmath value agrees with the longdouble value to every displayed digit,
so the three arithmetics coincide and the float64 numbers quoted
elsewhere in the paper are the numbers of record.

**The standing columns restate E2 on certified arithmetic.** Mates sit at
the 0.00 to 6.37 percentile of the census pairwise-L1 scale and at
2.5e-04 to 0.22 of the census median, which reproduces the known global
smallness (the moment axis dominates global standing; the count-identical
class analysis remains the corrective, and E2C does not replace it). The
n=14 showcase pair (201, 203) lands at the 5.82 percentile, consistent
with the recorded 0.0 to 5.8 range. The no-advantage discipline survives
certification intact: the smallest separation is now certified real at
1.9e-08 and certified unmeasurable behind the 10^10 to 10^14 shot wall in
the same breath.

**Two observations for the lemma file, carried as observations only.**
First, the separations band: the bulk runs 1.9e-08 to 1.4e-06, and then a
five-pair cluster sits an order of magnitude above it at 1.677e-05 to
1.692e-05, with four n=16 pairs and the n=14 pair (201, 203) landing
within one percent of each other. A common leading-order mechanism for
the loud mates is suggested and not claimed. Second, the triple
(582, 3061, 3800) has near/far structure: 582 and 3061 separate at
4.86e-08 while both pairs involving 3800 separate at ~1.15e-06, a 24×
asymmetry inside one cospectral class.

**Head/tail decomposition.** The pre-registered expectation holds once it is
calibrated against the uniform baseline, and the refinement it needed is more
interesting than the expectation. A separation spread uniformly over the
sorted vector would put hf50 at 0.0031 of a pair's L1 at n=14 and 0.0008 at
n=16, with k50 near the middle of the vector. The observed table instead puts
90% of every pair's separation within the top 7% of coordinates (the worst
k90 is 2804 of 65536 at n=16 and 1133 of 16384 at n=14), so every residue
lives in the head region and none lives in the tail. Within that universal
concentration, the profiles form families rather than a continuum. The loud
cluster is maximally head-weighted and internally near-identical: hf90 runs
0.848 to 0.856 and k50 runs 42 to 51 across all five pairs, which strengthens
the common-mechanism observation above (pair (1, 5) prints hf50 = 0.464 only
because its cumulative curve crosses one-half at coordinate 51, one past the
hf50 mark). The quieter pairs concentrate at fixed depths that recur across
unrelated pairs: nine pairs spanning L1 2.6e-07 to 3.9e-07 share k50 between
448 and 456, and ten pairs spanning 1.05e-06 to 1.16e-06 share k50 between
203 and 213, with hf50 as low as 0.018 in the latter family. Recurring depths
across unrelated cospectral pairs suggest that the residue concentrates at
specific plateaus of the sorted vector; the observation is logged for the
lemma and claimed as nothing more. The triple follows the family structure:
its near pair (582, 3061) carries a mid-depth profile (hf50 = 0.180,
k50 = 332), while both of its far pairs sit in the depth-210 family. One
consequence for the truncation framing deserves a sentence: a k=50 truncated
readout retains 54 to 67 percent of the separation for the head-weighted
families but as little as 1.8 percent for the fixed-depth families, so the
tracer's finding that truncation preferentially retains the residue is
pair-dependent rather than universal, which refines the two-of-three result
already on record instead of contradicting it.

**Scope.** E2C certifies precision and nothing else. It feeds the paper
exactly one sentence (the ×floor result) plus the certified restatement
of numbers other sections already own.